[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# LISTEN and NOTIFY &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with `send`, `heard` and `listening`. Run it first. Every
wait below has a timeout, so no cell can hang, and the tasks can be run in any order.


In [1]:
import asyncio
import getpass
import os
import subprocess
import sys
import threading
import time
import warnings
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors, sql

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def send(channel, payload="", after=0.0):
    """Notify from another connection, optionally a moment from now."""
    def once():
        with psycopg.connect("dbname=guide", autocommit=True) as conn:
            conn.execute("SELECT pg_notify(%s, %s)", (channel, payload))

    if after:
        threading.Timer(after, once).start()
    else:
        once()


def heard(listener, seconds=3, count=1):
    """Wait for notifications, with a timeout, because a cell that waits forever is a hung notebook."""
    return [(note.channel, note.payload) for note in
            listener.notifies(timeout=seconds, stop_after=count)]


def subscribe(*channels):
    """A connection of its own, already listening, so no section hears another section's backlog."""
    conn = psycopg.connect("dbname=guide", autocommit=True)         # autocommit, always
    for channel in channels:
        conn.execute(sql.SQL("LISTEN {}").format(sql.Identifier(channel)))
    return conn


def listening(conn):
    """What the server thinks this connection asked for, which is not always what you typed."""
    return [row[0] for row in conn.execute("SELECT * FROM pg_listening_channels()").fetchall()]


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


**1.** One channel, one message.


In [2]:
listener = psycopg.connect("dbname=guide", autocommit=True)
listener.execute("LISTEN greetings")

send("greetings", "hello from another connection", after=0.2)
print("heard:", heard(listener, seconds=3))

listener.close()


heard: [('greetings', 'hello from another connection')]


`autocommit=True` is the line that makes it work, and the timeout is the line that makes it safe to
run in a notebook.


**2.** Two channels, and which one spoke.


In [3]:
listener = psycopg.connect("dbname=guide", autocommit=True)
listener.execute("LISTEN orders")
listener.execute("LISTEN refunds")
print("subscribed to:", listening(listener))

send("orders", "order 17", after=0.1)
send("refunds", "refund 4", after=0.2)
for channel, payload in heard(listener, seconds=3, count=2):
    print(f"  on {channel}: {payload}")

listener.close()


subscribed to: ['orders', 'refunds']
  on orders: order 17
  on refunds: refund 4


The channel comes back with every notification, so one connection and one loop can serve the whole
program and dispatch on the name.


**3.** Only on commit.


In [4]:
listener = psycopg.connect("dbname=guide", autocommit=True)
listener.execute("LISTEN slow_news")

writer = psycopg.connect("dbname=guide")                            # an ordinary transaction
writer.execute("SELECT pg_notify('slow_news', 'written but not committed')")
print("before the commit:", heard(listener, seconds=1))

writer.commit()
print("after the commit: ", heard(listener, seconds=3))

writer.close()
listener.close()


before the commit: []
after the commit:  [('slow_news', 'written but not committed')]


A notification is held with the transaction that made it, so a rollback takes it away with
everything else. That is what makes a trigger safe: you are never told about a row that did not
survive.


**4.** The same thing in asyncpg.


In [5]:
conn = await asyncpg.connect(database="guide")
arrived = asyncio.Queue()

await conn.add_listener("async_news", lambda con, pid, ch, payload: arrived.put_nowait(payload))

send("async_news", "delivered by the event loop", after=0.2)
try:
    async with asyncio.timeout(3):
        print("heard:", await arrived.get())
except TimeoutError:
    print("nothing arrived")

await conn.close()


heard: delivered by the event loop


No autocommit to remember, because asyncpg never opened a transaction. The `asyncio.Queue` is what
gets a value out of a callback that cannot be awaited.


**5.** Capital letters, and where they go.


In [6]:
listener = psycopg.connect("dbname=guide", autocommit=True)
listener.execute("LISTEN Upper")
print("typed LISTEN Upper, subscribed to:", listening(listener))

send("Upper", "sent with the capital")
print("  pg_notify('Upper'):", heard(listener, seconds=1))

send("Upper", "sent with the capital", after=0.1)
send("upper", "sent folded", after=0.2)
print("  pg_notify('upper'):", heard(listener, seconds=3))

listener.close()


typed LISTEN Upper, subscribed to: ['upper']
  pg_notify('Upper'): []
  pg_notify('upper'): [('upper', 'sent folded')]


An unquoted identifier in a statement folds to lower case. A string handed to a function does not.
Lower case channel names with underscores make the whole question go away.


**6.** As big as it goes.


In [7]:
listener = psycopg.connect("dbname=guide", autocommit=True)
listener.execute("LISTEN sized")

biggest = 0
for size in (1000, 7998, 7999, 8000, 9000):
    try:
        send("sized", "x" * size)
        biggest = size
        print(f"  {size:5}: accepted")
    except errors.InvalidParameterValue:
        print(f"  {size:5}: payload string too long")

print("the largest that fits:", biggest)
heard(listener, seconds=1, count=3)
listener.close()


   1000: accepted
   7998: accepted
   7999: accepted
   8000: payload string too long
   9000: payload string too long
the largest that fits: 7999


Eight thousand bytes including the terminator. A payload is a key or an identifier, and anything
longer belongs in a row the listener goes and reads.


---

&#8592; **Back to:** [LISTEN and NOTIFY](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/15-listen-and-notify.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
